# Sample-data restart study

This notebook is a standalone audit of the public five-minute sample. It compares Smart Tagging restart context with observable ball physics and the public H/A/D timeline. Public labels are used only for validation; no label is fitted or written back into the pipeline.

The timing chain is: tagged restart → acceleration/kick proxy → flight or contest → first public H/A → comfortable-anchor evidence. Colours follow the main study: H red, A blue, D grey. No fixed hold duration, distance cutoff, or short/long rule is used here; values are displayed for observation and comparison.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'possession').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from possession.baseline_pipeline import ClosestTeamRuleModel
from possession.calculations.features import build_engineered_features
from possession.temporal_rules import load_rules
from starter.load_smart_tagging import load as load_smart_tagging

PUBLIC_START, PUBLIC_END = 11185, 18684
gt = pd.read_csv(ROOT / 'evaluation' / 'gt' / 'BAR-ATH_possession_gt_public.csv')
gt_by_frame = gt.set_index('frame')['label']
features = build_engineered_features(model=ClosestTeamRuleModel(), rules=load_rules())
public = features.loc[features.frame.between(PUBLIC_START, PUBLIC_END)].copy()
public['gt_label'] = gt_by_frame.reindex(public.frame).to_numpy()
print('public frames:', len(public), '| tagged sample rows:', len(load_smart_tagging()))

## Restart timing audit

Each raw SET PIECES interval is measured independently. The kick proxy is the frame of maximum positive ball-speed slope after the tag begins. The table also reports the first public H/A frame and the first available stable-anchor frame after the proxy.

In [ ]:
events = []
tags = load_smart_tagging()
for tag in tags:
    if str(tag.get('code', '')).upper() != 'SET PIECES': continue
    meta = tag.get('tags', {}) or {}
    restart_type = str(meta.get('Type', 'UNKNOWN'))
    tag_start = int(tag.get('start_frame', tag.get('start', -1)))
    tag_end = int(tag.get('end_frame', tag.get('end', tag_start)))
    if tag_end < PUBLIC_START or tag_start > PUBLIC_END: continue
    window = public.loc[public.frame.between(max(PUBLIC_START, tag_start), min(PUBLIC_END, tag_end))].copy()
    if window.empty: continue
    accel = pd.to_numeric(window.get('ball_acceleration_mps2', pd.Series(np.nan, index=window.index)), errors='coerce')
    kick_frame = int(window.loc[accel.idxmax(), 'frame']) if accel.notna().any() else np.nan
    gt_after = gt_by_frame.reindex(range(max(PUBLIC_START, tag_start), PUBLIC_END + 1))
    ha = gt_after.loc[gt_after.isin(['H', 'A'])]
    first_ha = int(ha.index[0]) if len(ha) else np.nan
    anchor = public.loc[(public.frame >= (kick_frame if pd.notna(kick_frame) else tag_start)) & public.selected_anchor_team.isin(['H','A'])] if 'selected_anchor_team' in public else pd.DataFrame()
    anchor_frame = int(anchor.frame.iloc[0]) if not anchor.empty else np.nan
    events.append({'restart_type':restart_type, 'tag_start_frame':tag_start, 'tag_end_frame':tag_end, 'kick_proxy_frame':kick_frame, 'kick_proxy_speed_mps':float(window.loc[window.frame.eq(kick_frame), 'ball_speed_derived_mps'].iloc[0]) if pd.notna(kick_frame) else np.nan, 'first_public_HA_frame':first_ha, 'seconds_tag_to_HA':(first_ha-tag_start)/25 if pd.notna(first_ha) else np.nan, 'first_anchor_frame':anchor_frame, 'seconds_kick_to_anchor':(anchor_frame-kick_frame)/25 if pd.notna(anchor_frame) and pd.notna(kick_frame) else np.nan})
restart_audit = pd.DataFrame(events)
display(restart_audit)
display(restart_audit.groupby('restart_type').agg(events=('tag_start_frame','size'), median_tag_to_HA_s=('seconds_tag_to_HA','median'), median_kick_to_anchor_s=('seconds_kick_to_anchor','median')).reset_index())

## Observed kick distance and landing/contest evidence

Distance is integrated from the candidate departure frame to the end of the observed event window. This is descriptive evidence, not an automatic short/long label and not a confirmed touch. The surrounding speed, acceleration, direction, density, and closing-speed traces must be inspected together.

In [ ]:
distance_rows = []
for event in events:
    kick = event['kick_proxy_frame']
    if pd.isna(kick): continue
    local = public.loc[public.frame.between(int(kick), min(PUBLIC_END, int(event['tag_end_frame']) + 100))].copy()
    # No fixed stop/short/long threshold is used. Keep the full observed
    # window so a reviewer can see whether speed falls, direction changes,
    # another acceleration appears, or players converge at the landing area.
    end = int(local.frame.iloc[-1])
    segment = local.loc[local.frame.between(int(kick), end)]
    distance_m = pd.to_numeric(segment.get('ball_displacement_m', pd.Series(0.0, index=segment.index)), errors='coerce').fillna(0.0).sum()
    landing = local.iloc[-1]
    distance_rows.append({**event, 'distance_end_frame':end, 'distance_m':float(distance_m), 'landing_speed_mps':float(pd.to_numeric(landing.ball_speed_derived_mps, errors='coerce')), 'landing_density_5m':float(pd.to_numeric(landing.landing_opponent_density_5m, errors='coerce')), 'landing_approach_speed_mps':float(pd.to_numeric(landing.landing_opponent_approach_speed_mps, errors='coerce')), 'landing_arrival_margin_s':float(pd.to_numeric(landing.landing_arrival_time_margin_s, errors='coerce')), 'landing_pass_options_15m':float(pd.to_numeric(landing.landing_pass_options_15m, errors='coerce'))})
distance_table = pd.DataFrame(distance_rows)
display(distance_table[['restart_type','kick_proxy_frame','distance_end_frame','distance_m','landing_speed_mps','landing_density_5m','landing_approach_speed_mps','landing_arrival_margin_s','landing_pass_options_15m','seconds_tag_to_HA','seconds_kick_to_anchor']])

fig, ax = plt.subplots(figsize=(12, 4))
for label, colour in {'H':'#d62728','A':'#1f77b4','D':'#999999'}.items():
    frames = public.loc[public.gt_label.eq(label), 'frame']
    ax.scatter(frames, np.full(len(frames), {'H':2,'A':1,'D':0}[label]), s=5, color=colour, label=label)
for _, row in distance_table.iterrows():
    ax.axvline(row.kick_proxy_frame, color='#e66101', alpha=.7)
    ax.text(row.kick_proxy_frame, 2.25, f"{row.restart_type}\n{row.distance_m:.1f} m", rotation=90, va='bottom', fontsize=8)
ax.set_yticks([0,1,2], ['D','A','H']); ax.set_xlabel('frame'); ax.set_title('Public H/A/D timeline with restart kick proxies and distances'); ax.legend(); plt.tight_layout(); plt.show()